# TotalSegmentator v201 预处理（Colab）

与 `colab_abdomenct1k_preprocess.ipynb`、`preprocess/proc_spatial_sequence.py` **相同的处理逻辑与超参**：
- 每个 nii volume：**50 个 npy**（z≥128 时）或 **33 个 npy**（z<128 时，`int(50/1.5)`）
- patch **128³**，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`

**数据**：仅处理各病例目录下的 **`ct.nii.gz`（CT）**；**不处理** `segmentations/` 内标签（与 TCIA/RibFrac 仅对体积做预训练预处理一致）。数据根为 `unzip/Totalsegmentator_dataset_v201/`。

**输出**：所有 `.npy` 扁平放在与 `unzip/` **平级**的 **`npy/`** 下；文件名为 `s0000_ct_0_1.npy` 形式（`path[-2]` 病例 ID + `ct` + patch 序号）。

In [1]:
# Cell 1: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2: 安装依赖
!pip install -q nibabel monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 25.3 MB/s eta 0:00:00


In [3]:
# Cell 3: TotalSegmentator 预处理（与 TCIA / proc_spatial_sequence 逻辑一致）
import os
import random
import numpy as np
import nibabel as nib
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

DRIVE_UNZIP = '/content/drive/MyDrive/dataset/pretrain/TotalSegmentator/unzip'
TS_DATA_ROOT = os.path.join(DRIVE_UNZIP, 'Totalsegmentator_dataset_v201')
DRIVE_SAVE_ROOT = '/content/drive/MyDrive/dataset/pretrain/TotalSegmentator/npy'

PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode='trilinear'),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume_ts(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size):
    image, affine = load_nii_data(image_file)
    path_parts = image_file.replace('\\', '/').split('/')
    ds_name = path_parts[-2] if len(path_parts) >= 2 else 'case'
    nii_id = path_parts[-1].split('.nii.gz')[0].split('.nii')[0].split('.mha')[0]

    if len(image.shape) == 4:
        return []
    images = [image]
    n, patch_path_list = 0, []

    for image_index, image in enumerate(images):
        image = image.transpose((2, 1, 0))
        _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
        for i in range(_patch_num):
            n += 1
            patch_size = random.choice(patch_size_list)
            image_patch, _ = cut_patch(image, patch_size)
            image_patch = image_patch.transpose((2, 1, 0))
            image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
            save_name = os.path.join(save_root, '%s_%s_%s_%d.nii.gz' % (ds_name, nii_id, image_index, start_index + n))
            patch_path_list.append(save_name)
            np.save(save_name.replace('.nii.gz', '.npy'), image_patch)
    return patch_path_list


def collect_ts_ct_nii_files(data_root):
    """仅各病例目录下的 ct.nii.gz；跳过 segmentations 与 meta 等。"""
    out = []
    if not os.path.isdir(data_root):
        return out
    for dirpath, _, files in os.walk(data_root):
        parts = dirpath.replace('\\', '/').split('/')
        if '__MACOSX' in parts:
            continue
        if 'segmentations' in parts:
            continue
        for f in files:
            low = f.lower()
            if low == 'ct.nii.gz' or low == 'ct.nii':
                out.append(os.path.join(dirpath, f))
    return sorted(out)


os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
data_list = collect_ts_ct_nii_files(TS_DATA_ROOT)
print(f'数据根: {TS_DATA_ROOT}')
print(f'找到 {len(data_list)} 个 ct 体积（仅 ct.nii.gz，不含 segmentations）')

if len(data_list) == 0:
    print('未找到文件，请检查是否已解压至 unzip/Totalsegmentator_dataset_v201/。')
else:
    patch_list_all = []
    for i, path in enumerate(tqdm(data_list, desc='预处理')):
        patch_list = cut_and_save_one_volume_ts(path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE)
        patch_list_all += patch_list
        if (i + 1) % 50 == 0:
            print(f'[{i+1}/{len(data_list)}] {path} -> {len(patch_list)} npy')
    print(f'\n完成! 共 {len(patch_list_all)} 个 npy, 输出: {DRIVE_SAVE_ROOT}')

数据根: /content/drive/MyDrive/dataset/pretrain/TotalSegmentator/unzip/Totalsegmentator_dataset_v201
找到 1228 个 ct 体积（仅 ct.nii.gz，不含 segmentations）


预处理:   4%|▍         | 50/1228 [09:27<3:32:19, 10.81s/it]

[50/1228] /content/drive/MyDrive/dataset/pretrain/TotalSegmentator/unzip/Totalsegmentator_dataset_v201/s0065/ct.nii.gz -> 50 npy


预处理:   8%|▊         | 100/1228 [18:09<3:11:38, 10.19s/it]

[100/1228] /content/drive/MyDrive/dataset/pretrain/TotalSegmentator/unzip/Totalsegmentator_dataset_v201/s0123/ct.nii.gz -> 50 npy


...

[1150/1228] /content/drive/MyDrive/dataset/pretrain/TotalSegmentator/unzip/Totalsegmentator_dataset_v201/s1341/ct.nii.gz -> 33 npy


预处理:  98%|█████████▊| 1200/1228 [3:30:25<05:21, 11.48s/it]

[1200/1228] /content/drive/MyDrive/dataset/pretrain/TotalSegmentator/unzip/Totalsegmentator_dataset_v201/s1400/ct.nii.gz -> 50 npy


预处理: 100%|██████████| 1228/1228 [3:35:00<00:00, 10.51s/it]


完成! 共 58000 个 npy, 输出: /content/drive/MyDrive/dataset/pretrain/TotalSegmentator/npy


In [ ]:
# Cell 4 (可选): 生成 train_patch_spatial.txt 用于 pretrain
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), 'pretrain_lists')
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith('.npy')])
with open(os.path.join(list_save_dir, 'train_patch_spatial.txt'), 'w') as f:
    f.write('\n'.join(npy_files))
print(f'train_patch_spatial.txt 已保存: {list_save_dir}')